In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Configuração de dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

## Preprocessing

In [ ]:
FEATURE_DIM = 64 # Ex: 63 dimensões para texto (NLP) + 1 dimensão para tempo

def preprocess_sessions_to_tensor(df):
    sessions = df['session_id'].unique()
    X_list = []
    y_list = []
    
    for session in sessions:
        session_df = df[df['session_id'] == session].copy()
        session_df = session_df.sort_values('datetime')
        
        # 1. Feature Temporal: Diferença de tempo entre requisições (Delta T)
        time_diff = session_df['datetime'].diff().dt.total_seconds().fillna(0).values
        # Normalização simples do tempo
        time_diff_norm = (time_diff - np.mean(time_diff)) / (np.std(time_diff) + 1e-5)
        time_tensor = torch.tensor(time_diff_norm, dtype=torch.float32).unsqueeze(1) # (T, 1)
        
        # 2. Features Semânticas (NLP): Substitua o bloco abaixo pelo seu tokenizador/embedding
        # Aqui simulamos um embedding de 63 dimensões gerado a partir do texto
        nlp_tensor = torch.randn(len(session_df), FEATURE_DIM - 1) # (T, 63)
        
        # 3. Fusão Inicial (Concatenação)
        session_features = torch.cat([nlp_tensor, time_tensor], dim=1) # (T, 64)
        X_list.append(session_features)
        
        # Rótulo da sessão
        y_list.append(session_df['is_bot_session'].iloc[0])
        
    # Empilhando para criar o Batch
    X = torch.stack(X_list).to(device) # Shape: (Batch, T, Feature_Dim)
    y = torch.tensor(y_list, dtype=torch.float32).to(device) # Shape: (Batch,)
    
    return X, y

X_train, y_train = preprocess_sessions_to_tensor(df)
print(f"Shape de Entrada (Batch, T, D): {X_train.shape}")
print(f"Shape dos Rótulos (Batch): {y_train.shape}")